## Transformación


Una vez obtenidos los datos, es necesario transformarlos para que sean
consistentes y estén listos para su análisis. Esto incluye:

- Limpieza de datos: Eliminar registros duplicados, corregir errores
  tipográficos, valores atípicos y manejar valores faltantes.
- Normalización: Asegurar que los datos estén en un formato uniforme, como
  convertir todas las fechas a un formato estándar.


In [1]:
import re

import curl_cffi
import orjson
import pandas as pd
from IPython.display import Markdown
from shapely.geometry import shape

from registro_de_incidencias.folder import (
    CLEAN_CSV_PATH,
    CSV_PATH,
    LIMA_GEOJSON_PATH,
)
from registro_de_incidencias.geojson import GeoJSON

In [2]:
df = pd.read_csv(CSV_PATH, sep=";", low_memory=False)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 219373 entries, 0 to 219372
Data columns (total 18 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   N° CASO                219373 non-null  int64  
 1   ORIGEN                 219373 non-null  str    
 2   ZONA                   219228 non-null  str    
 3   BASE DESCENTRALIZADA   219373 non-null  str    
 4   TURNO                  219373 non-null  str    
 5   TIPO DE CASO           219373 non-null  str    
 6   FECHA DEL CASO         219373 non-null  int64  
 7   HORA DEL CASO          219373 non-null  str    
 8   ATENCION AL CASO       219373 non-null  int64  
 9   HORA ATENCION AL CASO  219373 non-null  int64  
 10  LATITUD                219373 non-null  float64
 11  LONGITUD               219373 non-null  float64
 12  ESTADO                 219373 non-null  str    
 13  FECHA CORTE            219373 non-null  int64  
 14  DEPARTAMENTO           219373 non-null  str    

Observamos que hay 18 columnas, de las cuales solo 1 contiene valores nulos, se
procede a hacer un análisis en busca de tomar valores únicos.


In [3]:
# Current
count_rows = df.shape[0]
count_zona_na = df["ZONA"].isna().sum()
percentage_zona_na = (count_zona_na / count_rows) * 100

# Drop
df = df.dropna(subset=["ZONA"])
new_count_rows = df.shape[0]

Markdown(
    f"""\
Dado que los valores nulos en la columna **ZONA** representan el
${percentage_zona_na:.2f}\\%$ del total de filas, se decide eliminar las filas
con valores nulos en esta columna para asegurar la integridad de los datos en el
análisis posterior, dejando un total de ${new_count_rows}$ filas."""
)

Dado que los valores nulos en la columna **ZONA** representan el
$0.07\%$ del total de filas, se decide eliminar las filas
con valores nulos en esta columna para asegurar la integridad de los datos en el
análisis posterior, dejando un total de $219228$ filas.

Siguiendo con el tratamiento de esa columna, se observa que tiene valores
atípicos, los cuales, dado que son pocos y no se tiene información suficiente
para corregirlos, se decide eliminarlos del conjunto de datos.


In [4]:
# Constant
invalid_values = [
    "ZONA 1 / SECTOR 1-2 CIA CALLAO,clase518ZONA 2 / SECTOR 9 CIA INGUNZA",
    "ZONA 2 / SECTOR 9 CIA INGUNZA,clase517ZONA 2 / SECTOR 8 BOCANEGRA",
    "ZONA 3 / SECTOR 12 CIA MARQUEZ,clase519ZONA 3 / SECTOR 11 CIA OQUENDO",
]

# Drop
count_pre_drop_rows = df.shape[0]

df = df[~df["ZONA"].str.contains("|".join(invalid_values), regex=True)]

count_post_drop_rows = df.shape[0]

Markdown(
    f"""\
Se eliminaron las filas con valores inválidos en la columna **ZONA**, resultando
en la eliminación de ${count_pre_drop_rows - count_post_drop_rows}$ filas,
dejando un total de ${count_post_drop_rows}$ filas para el análisis posterior."""
)

Se eliminaron las filas con valores inválidos en la columna **ZONA**, resultando
en la eliminación de $4$ filas,
dejando un total de $219224$ filas para el análisis posterior.

Ahora, se filtran las coordenadas para asegurarnos que los casos efectivamente
estén dentro del distrito del Callao, eliminando los casos que no cumplan con
esta condición en un margen de $100$ metros. Para eso, los datos de latitud y
longitud se obtienen a partir del siguiente repositorio que nos facilita la
recolección de esta información a partir de las páginas del Estado:
<https://github.com/joseluisq/peru-geojson-datasets>.


In [5]:
# Download
if not LIMA_GEOJSON_PATH.exists():
    geojson_response = curl_cffi.get(
        "https://raw.githubusercontent.com/joseluisq/peru-geojson-datasets/refs/heads/master/lima_callao_distritos.geojson",
        stream=True,
    )

    geojson_response.raise_for_status()

    tmp_path = LIMA_GEOJSON_PATH.with_suffix(".tmp")

    with tmp_path.open("wb") as f:
        for chunk in geojson_response.iter_content(chunk_size=8192):
            f.write(chunk)

    tmp_path.rename(LIMA_GEOJSON_PATH)

# Load
with LIMA_GEOJSON_PATH.open() as f:
    peru_geojson = GeoJSON(**orjson.loads(f.read()))

callao_feature = next(
    (
        feature
        for feature in peru_geojson.features
        if feature.properties.distrito == "CALLAO"
    ),
    None,
)

if callao_feature is None:
    msg = "No se encontró el distrito del Callao en el GeoJSON."

    raise ValueError(msg)

# Shape
meters_margin = 100
grades_margin = meters_margin / 111_320
callao_shape = shape(callao_feature.geometry.model_dump()).buffer(grades_margin)

callao_left, callao_bottom, callao_right, callao_top = callao_shape.bounds

# Filter
count_pre_filter_rows = df.shape[0]

outside_lat = ~df["LATITUD"].between(callao_bottom, callao_top)
outside_lon = ~df["LONGITUD"].between(callao_left, callao_right)

df = df[~(outside_lat | outside_lon)]

count_post_filter_rows = df.shape[0]
diff_count_rows = count_pre_filter_rows - count_post_filter_rows
percentage_filtered = (diff_count_rows / count_pre_filter_rows) * 100

Markdown(
    f"""\
Se filtraron las filas con coordenadas fuera de los límites del distrito del
Callao, resultando en la eliminación de ${diff_count_rows}$ filas, lo que
representa el ${percentage_filtered:.2f}\\%$ del total, dejando un total de
${count_post_filter_rows}$ filas para el análisis posterior."""
)

Se filtraron las filas con coordenadas fuera de los límites del distrito del
Callao, resultando en la eliminación de $629$ filas, lo que
representa el $0.29\%$ del total, dejando un total de
$218595$ filas para el análisis posterior.

Ahora, procedemos a verificar la columna que debería contener valores únicos.


In [6]:
must_be_unique_columns = ["N° CASO"]
duplicated_columns = []


for column in must_be_unique_columns:
    text = []

    unique_values_count = df[column].nunique()
    has_unique_values = unique_values_count == len(df)

    if has_unique_values:
        text.append(f"- La columna **{column}** contiene valores únicos.")

        continue

    # Process the duplicated values
    text.append(f"- La columna **{column}** no contiene valores únicos.")

    # Show the unique values vs the total number of rows
    text.append(f"  - Valores únicos: ${unique_values_count}$")
    text.append(f"  - Total de filas: ${len(df)}$")
    text.append(f"  - Valores duplicados: ${len(df) - unique_values_count}$")
    text.append(
        f"  - Porcentaje de valores únicos: ${(unique_values_count / len(df)) * 100:.3f}\\%$"
    )

    # Display the duplicated rows for the column
    duplicated_rows = df[df.duplicated(subset=[column], keep=False)]
    text.append(
        f"\n{duplicated_rows[[*must_be_unique_columns, 'ORIGEN', 'ZONA', 'FECHA DEL CASO', 'HORA DEL CASO']].to_markdown(index=False)}"
    )

    duplicated_columns.append(column)

    display(Markdown("\n".join(text)))

df = df.drop_duplicates(subset=duplicated_columns)

- La columna **N° CASO** no contiene valores únicos.
  - Valores únicos: $218591$
  - Total de filas: $218595$
  - Valores duplicados: $4$
  - Porcentaje de valores únicos: $99.998\%$

|   N° CASO | ORIGEN   | ZONA                                 |   FECHA DEL CASO | HORA DEL CASO   |
|----------:|:---------|:-------------------------------------|-----------------:|:----------------|
|    801251 | CAMARA   | ZONA 1 / SECTOR 6 CIA DULANTO        |         10102025 | 10:35           |
|    801251 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |         10102025 | 10:35           |
|    856869 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |         29102025 | 17:42           |
|    856869 | CAMARA   | ZONA 3 / SECTOR 11 CIA OQUENDO       |         29102025 | 17:42           |
|    876165 | CAMARA   | ZONA 3 / SECTOR 11 CIA OQUENDO       |          4112025 | 1732            |
|    876165 | CAMARA   | ZONA 2 / SECTOR 9 CIA INGUNZA        |          4112025 | 1732            |
|    890838 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |          8112025 | 2107            |
|    890838 | CAMARA   | ZONA 1 / SECTOR 5 LA LEGUA           |          8112025 | 2107            |

Debido a que se tratan de casos donde se reportaron por más de una cámara, se
procede a eliminar debido a que al ser pocos no brindarían información relevante
al análisis. Teniendo en cuenta eso, se procede a normalizar los datos para
tener un formato uniforme para cada nombre propios, etc.


In [7]:
def to_title_case(_str: str) -> str:
    minor_words = {
        "de",
        "del",
        "la",
        "las",
        "los",
        "y",
        "a",
        "al",
        "en",
        "el",
        "un",
        "una",
        "uno",
    }
    mapped_words = {
        "QR": "QR",
        "CIA": "CIA",
        "BD": "BD",
        "TELEFONO": "Teléfono",
        "WHATSAPP": "WhatsApp",
        "CAMARA": "Cámara",
        "FISCALIZACION": "Fiscalización",
    }

    words = _str.split()
    result = []

    for i, word in enumerate(words):
        if word in mapped_words:
            result.append(mapped_words[word])
        elif word.startswith("C2"):
            match = re.match(r"C2(-?)(.+)", word)

            if match:
                result.append(f"C2 - {match.group(2).title()}")
            else:
                result.append(word)
        elif i == 0 or word.lower() not in minor_words:
            result.append(word.capitalize())
        else:
            result.append(word.lower())

    return " ".join(result)


title_case_columns = [
    "ORIGEN",
    "TURNO",
    "TIPO DE CASO",
    "BASE DESCENTRALIZADA",
]

for column in title_case_columns:
    df[column] = (
        df[column].astype(str).str.strip().str.upper().apply(to_title_case)
    )

text = []
for column in title_case_columns:
    unique_values = df[column].value_counts()
    text.append(f"- Columna **{column}** tiene los siguientes valores únicos:")
    for value, count in unique_values.items():
        text.append(f"  - `{value}`: ${count}$")
    text.append("")

Markdown("\n".join(text))

- Columna **ORIGEN** tiene los siguientes valores únicos:
  - `Cámara`: $213282$
  - `WhatsApp`: $3310$
  - `App Callao Seguro`: $540$
  - `Teléfono`: $453$
  - `Boton de Panico`: $400$
  - `Atencion Presencial`: $328$
  - `Radio`: $221$
  - `QR - Contacto Ciudadano`: $57$

- Columna **TURNO** tiene los siguientes valores únicos:
  - `Noche`: $59805$
  - `Madrugada`: $58671$
  - `Tarde`: $53924$
  - `Mañana`: $46191$

- Columna **TIPO DE CASO** tiene los siguientes valores únicos:
  - `Transito y Seguridad Vial`: $141200$
  - `Ambientales`: $36702$
  - `Fiscalización y Defensa Civil`: $21751$
  - `Apoyo al Ciudadano`: $16621$
  - `Seguridad Ciudadana`: $1700$
  - `App Callao Seguro`: $519$
  - `Protección Familiar`: $33$
  - `Salud`: $33$
  - `Casos Especiales`: $28$
  - `Ciudadano`: $4$

- Columna **BASE DESCENTRALIZADA** tiene los siguientes valores únicos:
  - `C - Oquendo`: $49829$
  - `C - Ramon Castilla`: $32881$
  - `C - Quilca`: $26333$
  - `C - Tomas Valle`: $22918$
  - `C - Obelisco`: $18266$
  - `C2 - Quilca`: $14214$
  - `Base Central de Monitoreo`: $12733$
  - `C2 - Bocanegra`: $9676$
  - `C2 - Ramón Castilla`: $8633$
  - `C2 - Obelisco`: $7157$
  - `C2 - Tomas Valle`: $6782$
  - `C - Bocanegra`: $5246$
  - `BD Ovalo de las Americas`: $886$
  - `BD Parque los Pozos`: $677$
  - `BD Plaza de Armas Marquez`: $550$
  - `BD Mujica Gallo`: $330$
  - `BD Pacasmayo`: $212$
  - `BD Santa Marina Sur`: $200$
  - `Base Ovalo de las Americas`: $195$
  - `BD Parque del Ejercito`: $168$
  - `BD Elmer Faucett`: $162$
  - `BD Alejandro Bertello`: $154$
  - `BD Alfredo Palacios`: $93$
  - `BD Nuevo Aeropuerto`: $93$
  - `BD los Dominicos`: $66$
  - `Base Costanera`: $40$
  - `BD Ovalo Centenario`: $30$
  - `BD Santa Rosa`: $21$
  - `BD Plaza Santa Rosa`: $12$
  - `BD Parque 10 de Junio`: $7$
  - `Base Bertello`: $7$
  - `Base Sin Fronteras`: $6$
  - `Base Plaza Marquez`: $3$
  - `Base Transporte`: $2$
  - `Base Dominicos`: $2$
  - `Base Manuel Mujica`: $2$
  - `Base Parque de Junio`: $2$
  - `Base Parque los Pozos`: $1$
  - `Base Plaza Santa Rosa`: $1$
  - `Base Pacasmayo`: $1$


Teniendo en cuenta el contexto de los datos, confirmamos que los registros
provienen efectivamente del departamento, provincia y distrito del Callao, así
que procedemos a eliminar las columnas de `DEPARTAMENTO`, `PROVINCIA`,
`DISTRITO` y `UBIGEO` para evitar redundancia en el análisis. Además, se
eliminará la columna `FECHA CORTE`, ya que contiene la fecha en la que se
realizó el corte de datos, lo cual no es relevante para el análisis de las
incidencias. Del mismo modo, la columna de `ESTADO` se eliminará debido a que
todos los registros tienen el mismo valor (`CERRADO`), lo cual no aporta
información relevante para el análisis.


In [8]:
df = df.drop(
    columns=[
        "DEPARTAMENTO",
        "PROVINCIA",
        "DISTRITO",
        "FECHA CORTE",
        "ESTADO",
        "UBIGEO",
    ]
)

Ahora procedemos a separar la columna **ZONA** en dos columnas:

- `zone_number`: número de zona (entero)
- `sector_name`: nombre del sector (texto)

El formato original es `ZONA X / SECTOR Y NOMBRE`, por ejemplo:
`ZONA 1 / SECTOR 1-2 CIA CALLAO`.


In [9]:
def parse_zone(zone: str) -> tuple[int, str]:
    match = re.match(
        r"ZONA\s+(\d+)\s*/\s*SECTOR\s+[\d\-]+\s*(.*)",
        zone,
    )
    if match:
        zone_number = int(match.group(1))
        sector_name = match.group(2).strip().title().replace("Cia", "CIA")
        return zone_number, sector_name

    return 0, zone


df["ZONA PARSED"] = df["ZONA"].apply(parse_zone)
df["ZONA"] = df["ZONA PARSED"].apply(lambda x: x[0])
df["SECTOR"] = df["ZONA PARSED"].apply(lambda x: x[1])
df = df.drop(columns=["ZONA PARSED"])

Markdown(
    f"""\
Columnas resultantes después de separar **ZONA**:

- `ZONA`: valores únicos = ${df["ZONA"].nunique()}$
- `SECTOR`: valores únicos = ${df["SECTOR"].nunique()}$

Primeras filas:

{df[["ZONA", "SECTOR"]].head().to_markdown(index=False)}"""
)

Columnas resultantes después de separar **ZONA**:

- `ZONA`: valores únicos = $3$
- `SECTOR`: valores únicos = $11$

Primeras filas:

|   ZONA | SECTOR      |
|-------:|:------------|
|      1 | CIA Callao  |
|      2 | CIA Ingunza |
|      1 | CIA Callao  |
|      1 | CIA Callao  |
|      2 | CIA Ingunza |

Ahora procederemos a hacer la transformación de los datos de fecha y hora a su
formato correcto, para esto se utilizará la función `pd.to_datetime` de la
librería `pandas` para convertir las columnas `FECHA DEL CASO` y
`ATENCION AL CASO` a su formato de fecha, y lo mismo para `HORA DEL CASO` y
`HORA ATENCION AL CASO` para su formato de hora.


In [10]:
df["FECHA DEL CASO"] = pd.to_datetime(
    df["FECHA DEL CASO"].astype(str).str.zfill(8),
    errors="coerce",
    format="%d%m%Y",
)
df["HORA DEL CASO"] = pd.to_datetime(
    df["HORA DEL CASO"].str.replace(":", "").str.zfill(4),
    errors="coerce",
    format="%H%M",
).dt.time
df["ATENCION AL CASO"] = pd.to_datetime(
    df["ATENCION AL CASO"].astype(str).str.zfill(8),
    errors="coerce",
    format="%d%m%Y",
)
df["HORA ATENCION AL CASO"] = pd.to_datetime(
    df["HORA ATENCION AL CASO"].astype(str).str.zfill(4),
    errors="coerce",
    format="%H%M",
).dt.time

Finalmente, se muestra un resumen de las columnas con su tipo de dato, cantidad
de valores nulos y no nulos, para verificar que la transformación se haya
realizado correctamente. El resultado final se exportará a un nuevo archivo CSV
para el proceso de carga.


In [11]:
df = df.rename(
    columns={
        "N° CASO": "n_caso",
        "ORIGEN": "origin",
        "ZONA": "zone",
        "BASE DESCENTRALIZADA": "decentralized_base",
        "TURNO": "shift",
        "TIPO DE CASO": "category",
        "FECHA DEL CASO": "case_date",
        "HORA DEL CASO": "case_time",
        "ATENCION AL CASO": "case_attention",
        "HORA ATENCION AL CASO": "case_attention_time",
        "LATITUD": "latitude",
        "LONGITUD": "longitude",
        "SECTOR": "sector",
    }
)
df.to_csv(CLEAN_CSV_PATH, index=False)


resumen = pd.DataFrame({
    "Columna": df.columns,
    "Tipo": df.dtypes.array,
    "No nulos": df.notna().sum().array,
    "Nulos": df.isna().sum().array,
})

Markdown(resumen.to_markdown(index=False))

| Columna             | Tipo           |   No nulos |   Nulos |
|:--------------------|:---------------|-----------:|--------:|
| n_caso              | int64          |     218591 |       0 |
| origin              | str            |     218591 |       0 |
| zone                | int64          |     218591 |       0 |
| decentralized_base  | str            |     218591 |       0 |
| shift               | str            |     218591 |       0 |
| category            | str            |     218591 |       0 |
| case_date           | datetime64[us] |     218591 |       0 |
| case_time           | object         |     218591 |       0 |
| case_attention      | datetime64[us] |     218591 |       0 |
| case_attention_time | object         |     218591 |       0 |
| latitude            | float64        |     218591 |       0 |
| longitude           | float64        |     218591 |       0 |
| sector              | str            |     218591 |       0 |